In [6]:
import pandas as pd
import numpy as np
import jellyfish
import re
from xgboost import XGBClassifier
from sklearn.model_selection import GroupShuffleSplit
from concurrent.futures import ProcessPoolExecutor
import time
import gc
import json
import joblib
import warnings

warnings.filterwarnings('ignore')

# ==============================================================================
# 1. PARALLEL FEATURE WORKER (10 High-Precision Features)
# ==============================================================================
def compute_pairwise_features(df_chunk):
    df_chunk = df_chunk.copy()
    
    # Text Arrays
    n1 = df_chunk['clean_name_s1'].fillna('').astype(str).values
    n2 = df_chunk['clean_name_tgt'].fillna('').astype(str).values
    t1 = df_chunk['clean_name_s1'].fillna('').astype(str).values
    t2 = df_chunk['Trans_name_tgt'].fillna('').astype(str).values
    a1 = df_chunk['norm_address_s1'].fillna('').astype(str).values
    a2 = df_chunk['norm_address_tgt'].fillna('').astype(str).values
    c1 = df_chunk['country_s1'].fillna('').astype(str).values
    c2 = df_chunk['country_tgt'].fillna('').astype(str).values

    n_rows = len(df_chunk)
    
    # Feature Arrays
    jaro_name = np.zeros(n_rows, dtype=np.float32)
    trans_jaro = np.zeros(n_rows, dtype=np.float32)
    lev_address = np.zeros(n_rows, dtype=np.float32)
    country_match = np.zeros(n_rows, dtype=np.int8)
    len_diff_name = np.zeros(n_rows, dtype=np.int32)
    exact_name_match = np.zeros(n_rows, dtype=np.int8)
    name_jaccard = np.zeros(n_rows, dtype=np.float32)
    addr_jaccard = np.zeros(n_rows, dtype=np.float32)
    addr_missing = np.zeros(n_rows, dtype=np.int8)
    addr_digit_match = np.zeros(n_rows, dtype=np.int8)

    # Pre-compile regex for fast digit extraction
    digit_pattern = re.compile(r'\d+')

    for i in range(n_rows):
        s1_n, tgt_n = n1[i], n2[i]
        addr1, addr2 = a1[i], a2[i]
        
        # 1. Exact Name Signal
        exact_name_match[i] = 1 if (s1_n == tgt_n and s1_n != "") else 0
        
        # 2. String Distances
        jaro_name[i] = jellyfish.jaro_winkler_similarity(s1_n, tgt_n)
        trans_jaro[i] = jellyfish.jaro_winkler_similarity(t1[i], t2[i])
        len_diff_name[i] = abs(len(s1_n) - len(tgt_n))
        
        # 3. Name Token Jaccard
        s1_toks = set(s1_n.split())
        tgt_toks = set(tgt_n.split())
        if s1_toks and tgt_toks:
            name_jaccard[i] = len(s1_toks & tgt_toks) / len(s1_toks | tgt_toks)
            
        # 4. Address Logic
        addr_missing[i] = 1 if (addr1 == "" or addr2 == "") else 0
        
        max_l = max(len(addr1), len(addr2))
        if max_l > 0:
            lev_address[i] = 1.0 - (jellyfish.levenshtein_distance(addr1, addr2) / max_l)
            
        # 5. Address Token Jaccard
        a1_toks = set(addr1.split())
        a2_toks = set(addr2.split())
        if a1_toks and a2_toks:
            addr_jaccard[i] = len(a1_toks & a2_toks) / len(a1_toks | a2_toks)
            
        # 6. Address Digit Match (Branch collision detection)
        nums1 = set(digit_pattern.findall(addr1))
        nums2 = set(digit_pattern.findall(addr2))
        if nums1 and nums2:
            addr_digit_match[i] = 1 if (nums1 & nums2) else -1
            
        # 7. Country Match
        if c1[i] == c2[i] and c1[i] != "":
            country_match[i] = 1

    # Attach features to DataFrame
    df_chunk['jaro_name'] = jaro_name
    df_chunk['trans_jaro'] = trans_jaro
    df_chunk['lev_address'] = lev_address
    df_chunk['country_match'] = country_match
    df_chunk['len_diff_name'] = len_diff_name
    df_chunk['exact_name_match'] = exact_name_match
    df_chunk['name_jaccard'] = name_jaccard
    df_chunk['addr_jaccard'] = addr_jaccard
    df_chunk['addr_missing'] = addr_missing
    df_chunk['addr_digit_match'] = addr_digit_match
    
    return df_chunk

# ==============================================================================
# 2. DATA LOADING & GROUND TRUTH LABELING
# ==============================================================================
if __name__ == '__main__':
    print("Loading Candidates & Raw Data...")
    candidates = pd.read_csv("strict_s1_candidate_pairs.csv")
    
    cols = ['entity_id', 'clean_name', 'Trans_name', 'norm_address', 'country']
    s1 = pd.read_csv("../ML2026/cleaned_dataset/source1.tsv", sep='\t', usecols=cols)
    s2 = pd.read_csv("../ML2026/cleaned_dataset/source2.tsv", sep='\t', usecols=cols)
    s3 = pd.read_csv("../ML2026/cleaned_dataset/source3.tsv", sep='\t', usecols=cols)
    
    target_df = pd.concat([s2, s3], ignore_index=True)
    del s2, s3
    gc.collect()

    print("Loading Ground Truth...")
    ground_truth = pd.read_csv("../ML2026/student_resource/dataset/train/train_ground_truth.tsv", sep="\t")
    
    ground_truth['matched_entity_ids'] = ground_truth['matched_entity_ids'].fillna('').str.split(',')
    gt_exploded = ground_truth.explode('matched_entity_ids')
    gt_exploded = gt_exploded[gt_exploded['matched_entity_ids'] != '']
    true_pairs = set(zip(gt_exploded['source1_entity_id'], gt_exploded['matched_entity_ids']))

    print("Generating Binary Labels...")
    candidate_tuples = list(zip(candidates['s1_entity_id'], candidates['target_entity_id']))
    candidates['label'] = [1 if pair in true_pairs else 0 for pair in candidate_tuples]
    
    del true_pairs, candidate_tuples, ground_truth, gt_exploded
    gc.collect()

    print(f"Total Pairs: {len(candidates):,} | Positives: {candidates['label'].sum():,} | Negatives: {(candidates['label']==0).sum():,}")

    # ==============================================================================
    # 3. MERGE METADATA & 192-CORE PARALLEL FEATURE EXTRACTION
    # ==============================================================================
    print("Merging Text Fields...")
    df = candidates.merge(s1, left_on='s1_entity_id', right_on='entity_id', how='left')
    df = df.merge(target_df, left_on='target_entity_id', right_on='entity_id', suffixes=('_s1', '_tgt'), how='left')
    
    del candidates, target_df, s1
    gc.collect()

    print("Computing Pairwise Features across 192 vCPUs...")
    start_feat = time.time()
    
    # 192 cores utilized fully

    from joblib import Parallel, delayed

    num_cores = 192  
    df_chunks = np.array_split(df, num_cores * 2)

    # joblib utilizes shared memory structures automatically
    results = Parallel(n_jobs=192, batch_size=1)(
        delayed(compute_pairwise_features)(chunk) for chunk in df_chunks
    )

    df = pd.concat(results, ignore_index=True)
    del df_chunks
    gc.collect()
    print(f"Feature computation completed in {time.time() - start_feat:.2f} seconds.")

    # ==============================================================================
    # 4. GROUPED SPLIT (Zero Data Leakage)
    # ==============================================================================
    features = [
        'jaro_name', 'trans_jaro', 'name_jaccard', 'exact_name_match', 'len_diff_name',
        'lev_address', 'addr_jaccard', 'addr_missing', 'addr_digit_match', 
        'country_match'
    ]

    print("Splitting Data (Grouped by S1 Entity ID)...")
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, val_idx = next(gss.split(df, groups=df['s1_entity_id']))

    train_data = df.iloc[train_idx]
    val_data = df.iloc[val_idx].copy()

    X_train, y_train = train_data[features], train_data['label']
    X_val, y_val = val_data[features], val_data['label']

    num_neg = (y_train == 0).sum()
    num_pos = (y_train == 1).sum()
    pos_weight = num_neg / num_pos if num_pos > 0 else 1.0

    # ==============================================================================
    # 5. XGBOOST MODEL TRAINING
    # ==============================================================================
    print("\nTraining Precision-Heavy XGBoost Classifier...")
    xgb_model = XGBClassifier(
        n_estimators=700,
        learning_rate=0.05,
        max_depth=7,
        subsample=0.8,
        colsample_bytree=0.9,
        scale_pos_weight=pos_weight,
        tree_method='hist',           
        n_jobs=192,                   # Utilizing all 192 cores for training
        random_state=42,
        eval_metric='logloss',
        early_stopping_rounds=50
    )

    xgb_model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=50
    )

    # ==============================================================================
    # 6. OPTIMAL F0.5 THRESHOLD CALIBRATION
    # ==============================================================================
    print("\nCalibrating F0.5 Threshold on Validation Set:")
    val_data['match_prob'] = xgb_model.predict_proba(X_val)[:, 1]

    best_f05 = 0
    best_thresh = 0.5

    for thresh in np.arange(0.70, 0.98, 0.02):
        pred_positive = (val_data['match_prob'] >= thresh).astype(int)
        
        tp = np.sum((pred_positive == 1) & (val_data['label'] == 1))
        fp = np.sum((pred_positive == 1) & (val_data['label'] == 0))
        fn = np.sum((pred_positive == 0) & (val_data['label'] == 1))
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        
        f05 = (1.25 * precision * recall) / ((0.25 * precision) + recall) if (precision + recall) > 0 else 0
        
        if f05 > best_f05:
            best_f05 = f05
            best_thresh = thresh

        print(f"Thresh: {thresh:.2f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F0.5: {f05:.4f}")

    print(f"\n--> Selected Optimal F0.5 Threshold: {best_thresh:.2f} (F0.5: {best_f05:.4f})")

    # ==============================================================================
    # 7. SAVE ARTIFACTS
    # ==============================================================================
    joblib.dump(xgb_model, 'xgboost_entity_resolution.pkl')
    with open('threshold_config.json', 'w') as f:
        json.dump({'best_threshold': float(best_thresh)}, f)
    print("\nXGBoost Model & Threshold saved successfully! Ready for Test Inference.")

Loading Candidates & Raw Data...
Loading Ground Truth...
Generating Binary Labels...
Total Pairs: 84,596,526 | Positives: 100,120 | Negatives: 84,496,406
Merging Text Fields...
Computing Pairwise Features across 192 vCPUs...
Feature computation completed in 348.45 seconds.
Splitting Data (Grouped by S1 Entity ID)...

Training Precision-Heavy XGBoost Classifier...
[0]	validation_0-logloss:0.64575
[50]	validation_0-logloss:0.05858
[100]	validation_0-logloss:0.02301
[150]	validation_0-logloss:0.01938
[200]	validation_0-logloss:0.01797
[250]	validation_0-logloss:0.01702
[300]	validation_0-logloss:0.01630
[350]	validation_0-logloss:0.01564
[400]	validation_0-logloss:0.01507
[450]	validation_0-logloss:0.01462
[500]	validation_0-logloss:0.01421
[550]	validation_0-logloss:0.01386
[600]	validation_0-logloss:0.01349
[650]	validation_0-logloss:0.01317
[699]	validation_0-logloss:0.01292

Calibrating F0.5 Threshold on Validation Set:
Thresh: 0.70 | Precision: 0.2932 | Recall: 0.9827 | F0.5: 0.3411
